# Indonesian Dict Builder — v3

Refactor of v2 to align with sanity check v4's per-resource tier system.

## What changed from v2

**Filter changed from overall tier to parcor tier.** v2 filtered Strategy B
on `overall_tier_postfix ∈ {good, needs_attention}`. v4 sanity checks
removed the `overall_tier` column entirely, so v3 vocab builder filters on
`parcor_tier_postfix ∈ {good, needs_attention}` instead.

This change is also semantically cleaner: this notebook pools Indonesian
content from the parcor resource, so parcor quality is the relevant
filter. Including billex/morph quality in the filter (as v2 implicitly did
through the "overall" aggregation) was conservative for no good reason —
billex problems shouldn't disqualify a dict's parcor content.

**Pool size will likely grow.** Under v4 sanity, only 6 dicts have broken
parcor (vs ~29 with broken overall under v2). Strategy B's pool grows
from ~37 dicts to ~62 dicts. This is intentional — more clean data is
better, and dicts that have broken billex/morph but clean parcor were
previously excluded for unrelated reasons.

**Strategy C threshold unchanged at ≥3 dicts.** With a larger pool, ≥3 is
easier to meet, so Strategy C will also grow. We keep the threshold to
match the original methodology; if needed, can be re-tuned later.

## Outputs (in `../spellCheckDicts/vocab_strategies_v3/`)

- `dict_strategy_original.json` — Strategy A: all 68 dicts
- `dict_strategy_B.json` — Strategy B: parcor good + needs_attention
- `dict_strategy_C.json` — Strategy C: B + token appears in ≥3 dicts
- `_strategy_comparison.csv` — token counts and overlap per strategy
- `_dict_inclusion.csv` — which dicts are included in each strategy

v2 outputs in `../spellCheckDicts/` left intact for comparison.

## 1. Imports & paths

In [8]:
import re
import json
from pathlib import Path
from collections import Counter, defaultdict
from typing import Optional

import pandas as pd
import numpy as np

from _common import parse_dict_id, load_direction_lookup, direction_for

PARCOR_DIR = Path("../Ekstraksi/11. Parallel Corpus - Fixed")
QUALITY_TIERS_PATH = Path("../csvAnalysis/sanity_v4/_quality_tiers.csv")
DST_DIR = Path("../spellCheckDicts/vocab_strategies_v3")
DST_DIR.mkdir(parents=True, exist_ok=True)

# Strategy C threshold (token must appear in this many distinct dicts)
STRATEGY_C_MIN_DICTS = 3

# Token cleaning rules
TOKEN_MIN_LENGTH = 2  # filter out single characters
TOKEN_REGEX = re.compile(r"[a-zA-Z]+")  # only alphabetic tokens

print(f"Parcor source:    {PARCOR_DIR}")
print(f"Quality tiers:    {QUALITY_TIERS_PATH}")
print(f"Output:           {DST_DIR.resolve()}")
print(f"Strategy C min dicts: {STRATEGY_C_MIN_DICTS}")
print(f"Token min length: {TOKEN_MIN_LENGTH}")

assert PARCOR_DIR.exists(), f"Parcor directory missing: {PARCOR_DIR}"
assert QUALITY_TIERS_PATH.exists(), (
    f"Quality tiers file missing: {QUALITY_TIERS_PATH}\n"
    f"Run Distribution Sanity Checks v4.ipynb first."
)

Parcor source:    ..\Ekstraksi\11. Parallel Corpus - Fixed
Quality tiers:    ..\csvAnalysis\sanity_v4\_quality_tiers.csv
Output:           C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\spellCheckDicts\vocab_strategies_v3
Strategy C min dicts: 3
Token min length: 2


## 2. Load quality tiers and determine dict inclusion

For each strategy, determine the set of dict IDs to include based on
parcor tier:

- **Strategy A (original):** all 68 dicts, regardless of tier
- **Strategy B:** dicts where parcor_tier_postfix ∈ {good, needs_attention}
- **Strategy C:** same dict pool as B (token-level filter applied later)

In [9]:
tiers_df = pd.read_csv(QUALITY_TIERS_PATH, dtype={"dict_id": str})
print(f"Loaded {len(tiers_df)} dict tier records")

# Strategy A: all dicts
strategy_A_dicts = set(tiers_df["dict_id"])

# Strategy B and C: parcor good or needs_attention
acceptable_parcor_tiers = {"good", "needs_attention"}
strategy_BC_dicts = set(
    tiers_df[tiers_df["parcor_tier_postfix"].isin(acceptable_parcor_tiers)]["dict_id"]
)

# Diagnostic: which dicts are excluded from B?
excluded_from_B = strategy_A_dicts - strategy_BC_dicts
print(f"\nDict pool sizes:")
print(f"  Strategy A: {len(strategy_A_dicts)} dicts (all)")
print(f"  Strategy B/C pool: {len(strategy_BC_dicts)} dicts")
print(f"  Excluded from B/C: {len(excluded_from_B)} dicts (parcor broken or not_applicable)")
if excluded_from_B:
    print(f"    Excluded: {', '.join(sorted(excluded_from_B, key=lambda s: int(s)))}")

# Save inclusion table
inclusion_rows = []
for dict_id in sorted(strategy_A_dicts, key=lambda s: int(s)):
    tier_row = tiers_df[tiers_df["dict_id"] == dict_id]
    parcor_tier = tier_row["parcor_tier_postfix"].iloc[0] if len(tier_row) else "missing"
    inclusion_rows.append({
        "dict_id": dict_id,
        "parcor_tier_postfix": parcor_tier,
        "in_strategy_A": dict_id in strategy_A_dicts,
        "in_strategy_B": dict_id in strategy_BC_dicts,
        "in_strategy_C_pool": dict_id in strategy_BC_dicts,
    })
inclusion_df = pd.DataFrame(inclusion_rows)
inclusion_df.to_csv(DST_DIR / "_dict_inclusion.csv", index=False)
print(f"\nWrote {DST_DIR / '_dict_inclusion.csv'}")

Loaded 68 dict tier records

Dict pool sizes:
  Strategy A: 68 dicts (all)
  Strategy B/C pool: 57 dicts
  Excluded from B/C: 11 dicts (parcor broken or not_applicable)
    Excluded: 5, 8, 11, 25, 26, 31, 50, 51, 67, 78, 84

Wrote ..\spellCheckDicts\vocab_strategies_v3\_dict_inclusion.csv


## 3. Token extraction from parcor files

For each dict, read the parcor file and extract Indonesian tokens from
the Indonesian-side column (kalimat_asal or kalimat_tujuan, depending on
the direction). Tokens are lowercased, filtered to alphabetic characters,
and de-duplicated per dict.

In [10]:
direction_lookup = load_direction_lookup()


def extract_tokens_from_parcor(dict_id: str) -> set:
    """Read parcor file, return set of Indonesian tokens."""
    files = list(PARCOR_DIR.glob(f"{dict_id}_*.csv"))
    if not files:
        return set()
    try:
        df = pd.read_csv(files[0])
    except Exception:
        return set()

    if "kalimat_asal" not in df.columns or "kalimat_tujuan" not in df.columns:
        return set()

    direction = direction_for(dict_id, direction_lookup)
    # direction = 1 means kata_asal is Indonesian; direction = 0 means kata_tujuan is Indonesian
    if direction == 1:
        ind_col = "kalimat_asal"
    elif direction == 0:
        ind_col = "kalimat_tujuan"
    else:
        return set()  # unknown direction, skip

    tokens = set()
    for sentence in df[ind_col].dropna().astype(str):
        for token in TOKEN_REGEX.findall(sentence.lower()):
            if len(token) >= TOKEN_MIN_LENGTH:
                tokens.add(token)
    return tokens


# Per-dict token sets
print("Extracting tokens per dict...")
per_dict_tokens = {}
for dict_id in sorted(strategy_A_dicts, key=lambda s: int(s)):
    tokens = extract_tokens_from_parcor(dict_id)
    per_dict_tokens[dict_id] = tokens

# Diagnostic
n_dicts_with_tokens = sum(1 for ts in per_dict_tokens.values() if len(ts) > 0)
total_unique = len(set().union(*per_dict_tokens.values()))
print(f"  Dicts with extracted tokens: {n_dicts_with_tokens} / {len(per_dict_tokens)}")
print(f"  Total unique tokens across all dicts: {total_unique:,}")

Extracting tokens per dict...
  Dicts with extracted tokens: 68 / 68
  Total unique tokens across all dicts: 190,882


## 4. Build the three strategies

- **Strategy A:** union of tokens from all 68 dicts
- **Strategy B:** union of tokens from dicts in `strategy_BC_dicts`
- **Strategy C:** subset of Strategy B where each token appears in ≥3 dicts

In [11]:
# Strategy A: union of all tokens
strategy_A_tokens = set()
for dict_id in strategy_A_dicts:
    strategy_A_tokens |= per_dict_tokens.get(dict_id, set())

# Strategy B: union of tokens from filtered dicts
strategy_B_tokens = set()
for dict_id in strategy_BC_dicts:
    strategy_B_tokens |= per_dict_tokens.get(dict_id, set())

# Strategy C: count how many dicts each token appears in (within B's pool)
token_dict_count = Counter()
for dict_id in strategy_BC_dicts:
    for token in per_dict_tokens.get(dict_id, set()):
        token_dict_count[token] += 1

strategy_C_tokens = {
    token for token, count in token_dict_count.items()
    if count >= STRATEGY_C_MIN_DICTS
}

print(f"Strategy A (all dicts):           {len(strategy_A_tokens):,} unique tokens")
print(f"Strategy B (parcor good+needsA):  {len(strategy_B_tokens):,} unique tokens")
print(f"Strategy C (B + ≥{STRATEGY_C_MIN_DICTS} dicts):       {len(strategy_C_tokens):,} unique tokens")

Strategy A (all dicts):           190,882 unique tokens
Strategy B (parcor good+needsA):  173,329 unique tokens
Strategy C (B + ≥3 dicts):       23,936 unique tokens


## 5. Build the JSON dict files

The output format mirrors the legacy `spellCheckDicts/dict.json` format:
a frequency-style dict where each token maps to its appearance count.
For Strategies A and B, count = number of dicts containing the token.
For Strategy C, count is the same (since C is a subset of B's tokens).

In [12]:
# Compute appearance counts across A's pool (for Strategy A's output)
strategy_A_counts = Counter()
for dict_id in strategy_A_dicts:
    for token in per_dict_tokens.get(dict_id, set()):
        strategy_A_counts[token] += 1

# Strategy B's counts come from token_dict_count (already computed above)
strategy_B_counts = {tok: token_dict_count[tok] for tok in strategy_B_tokens}

# Strategy C's counts: same as B's, but filtered
strategy_C_counts = {tok: token_dict_count[tok] for tok in strategy_C_tokens}


def write_dict_json(filename: str, counts: dict):
    path = DST_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(dict(sorted(counts.items())), f, ensure_ascii=False, indent=0)
    print(f"  Wrote {path} — {len(counts):,} entries")


print("Writing dict JSON files...")
write_dict_json("dict_strategy_original.json", dict(strategy_A_counts))
write_dict_json("dict_strategy_B.json", strategy_B_counts)
write_dict_json("dict_strategy_C.json", strategy_C_counts)

Writing dict JSON files...
  Wrote ..\spellCheckDicts\vocab_strategies_v3\dict_strategy_original.json — 190,882 entries
  Wrote ..\spellCheckDicts\vocab_strategies_v3\dict_strategy_B.json — 173,329 entries
  Wrote ..\spellCheckDicts\vocab_strategies_v3\dict_strategy_C.json — 23,936 entries


## 6. Strategy comparison

How do the strategies compare in size and overlap? This validates that:
- Strategy B is a subset of Strategy A (or close to it)
- Strategy C is a subset of Strategy B
- The size progression A > B > C makes sense

In [13]:
comparison_rows = []

# Per-strategy stats
for strategy_name, token_set in [
    ("A_original", strategy_A_tokens),
    ("B_parcor_filtered", strategy_B_tokens),
    ("C_B_plus_cross_dict", strategy_C_tokens),
]:
    comparison_rows.append({
        "strategy": strategy_name,
        "n_tokens": len(token_set),
        "fraction_of_A": round(len(token_set) / len(strategy_A_tokens), 3) if strategy_A_tokens else 0,
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(DST_DIR / "_strategy_comparison.csv", index=False)

print("=== Strategy Comparison ===\n")
print(comparison_df.to_string(index=False))

# Pairwise overlaps
print(f"\n=== Pairwise containment ===")
print(f"  B ⊆ A: {strategy_B_tokens.issubset(strategy_A_tokens)} "
      f"({len(strategy_B_tokens & strategy_A_tokens):,} of B's {len(strategy_B_tokens):,} are in A)")
print(f"  C ⊆ B: {strategy_C_tokens.issubset(strategy_B_tokens)} "
      f"({len(strategy_C_tokens & strategy_B_tokens):,} of C's {len(strategy_C_tokens):,} are in B)")
print(f"  C ⊆ A: {strategy_C_tokens.issubset(strategy_A_tokens)} "
      f"({len(strategy_C_tokens & strategy_A_tokens):,} of C's {len(strategy_C_tokens):,} are in A)")

# Tokens in A but not B (i.e., from broken-parcor dicts)
A_only = strategy_A_tokens - strategy_B_tokens
print(f"\n  Tokens in A but not B: {len(A_only):,} "
      f"(from {len(excluded_from_B)} parcor-broken/NA dicts)")

# Tokens in B but not C (i.e., appear in <3 dicts)
B_only = strategy_B_tokens - strategy_C_tokens
print(f"  Tokens in B but not C: {len(B_only):,} "
      f"(appear in <{STRATEGY_C_MIN_DICTS} dicts)")

=== Strategy Comparison ===

           strategy  n_tokens  fraction_of_A
         A_original    190882          1.000
  B_parcor_filtered    173329          0.908
C_B_plus_cross_dict     23936          0.125

=== Pairwise containment ===
  B ⊆ A: True (173,329 of B's 173,329 are in A)
  C ⊆ B: True (23,936 of C's 23,936 are in B)
  C ⊆ A: True (23,936 of C's 23,936 are in A)

  Tokens in A but not B: 17,553 (from 11 parcor-broken/NA dicts)
  Tokens in B but not C: 149,393 (appear in <3 dicts)


## 7. Comparison with v2 outputs (if available)

Side-by-side comparison: how did the strategies change between v2 and v3?

This cell looks for v2 outputs in `../spellCheckDicts/` and reports size
deltas. If v2 outputs aren't found, this cell is skipped.

In [14]:
V2_DIR = Path("../spellCheckDicts")

v2_files = {
    "A_original":          "dict_strategy_original.json",
    "B_parcor_filtered":   "dict_strategy_B.json",
    "C_B_plus_cross_dict": "dict_strategy_C.json",
}

if V2_DIR.exists():
    print("=== v2 vs v3 comparison ===\n")
    print(f"{'Strategy':<25} {'v2 tokens':>12} {'v3 tokens':>12} {'Δ':>10} {'% change':>10}")
    print("-" * 75)
    for strategy, fname in v2_files.items():
        v2_path = V2_DIR / fname
        v3_path = DST_DIR / fname

        if not v2_path.exists():
            print(f"{strategy:<25} {'(not found)':>12}")
            continue

        with open(v2_path) as f:
            v2_dict = json.load(f)
        with open(v3_path) as f:
            v3_dict = json.load(f)

        n_v2 = len(v2_dict)
        n_v3 = len(v3_dict)
        delta = n_v3 - n_v2
        pct = round(100 * delta / n_v2, 1) if n_v2 else 0
        print(f"{strategy:<25} {n_v2:>12,} {n_v3:>12,} {delta:>+10,} {pct:>+9}%")
else:
    print("(v2 directory not found; skipping comparison)")

=== v2 vs v3 comparison ===

Strategy                     v2 tokens    v3 tokens          Δ   % change
---------------------------------------------------------------------------
A_original                 (not found)
B_parcor_filtered          (not found)
C_B_plus_cross_dict        (not found)


## 8. Reading guide

Three things to check after running this notebook:

1. **Strategy B size went up vs v2.** Expected because the new filter is
   less restrictive (parcor only, not all three resources). If it went
   down, something is wrong with the filter logic.

2. **Strategy C size also went up.** Same reason — bigger B pool means
   more tokens have a chance to appear in ≥3 dicts.

3. **B ⊆ A and C ⊆ B should hold.** If either containment fails, there's
   a bug in the strategy definitions.

The new dict files in `../spellCheckDicts/vocab_strategies_v3/` are drop-in
replacements for the v2 files. Downstream notebooks (the spellcheck
detection notebooks for pyspellchecker, n-gram, and pattern miner) need
to be repointed at the new path:

  Old: `../spellCheckDicts/dict_strategy_*.json`
  New: `../spellCheckDicts/vocab_strategies_v3/dict_strategy_*.json`

You'll likely want to re-run the spellcheck comparison analysis after
this — flag rates and method agreement may shift since the underlying
vocabulary has changed. The methodology stays the same; only the numbers
update.